# QA retrieval: tài liệu cha và Notes trước reranker

Cùng một Knowledge Base và cùng mã nguồn `rag/`. Notebook này so sánh **đường cũ** (fusion → reranker → bổ sung graph vào gói căn cứ) với **đường thử nghiệm** (fusion → tài liệu cha/Notes thật → reranker). Không chia câu hỏi bằng mẫu `if/else` theo tầng HS. Nhánh truy vấn bỏ mã nghi vấn chỉ chạy khi câu hỏi thực sự chứa mã bị phủ định hoặc đặt nghi vấn.

Chọn chính sách bằng **320 train**. Chỉ sau khi ghi cấu hình đã chọn mới mở **150 test**; 50 câu cũ được báo riêng như tập chẩn đoán, 100 câu mới là tập xác nhận độc lập. Output so sánh nằm trong notebook, không xuất report rời.

In [1]:
import os
import sys
from pathlib import Path

os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")
candidates = [Path(os.environ["AIAGENT_ROOT"])] if "AIAGENT_ROOT" in os.environ else []
for parent in [Path.cwd(), *Path.cwd().parents]:
    candidates.extend([parent, parent / "apps/aiAgent", parent / "frappe-bench/apps/aiAgent"])
ROOT = next((p.resolve() for p in candidates if (p / "rag/data.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Set AIAGENT_ROOT to the absolute aiAgent folder")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [2]:
import time

import pandas as pd
from IPython.display import display
from IPython.utils.io import capture_output

from rag.data import config, digest, load_corpus, read, write_json
from rag.db import connect
from rag.embedding import Embedder
from rag.evaluation import (
    CONFIGURATIONS,
    QA_CHUNK_TYPES,
    branch_cache,
    discover_qa,
    qa_direct_coverage,
    qa_objective,
    sweep_qa_weights,
)
from rag.retrieval import Retriever

pd.set_option("display.max_colwidth", 140)

bundle = load_corpus()
conn = connect()
with capture_output():
    embedder = Embedder(config())
runtime = Retriever(conn, embedder)
assert runtime.snapshot == bundle["snapshot"]
print("Active snapshot:", runtime.snapshot, "profile:", runtime.profile)
print("Corpus records:", len(bundle["records"]))

Active snapshot: 1a67df40d28f8b07a668746c profile: 3a2ce6e6d726a0f053748717
Corpus records: 5356


## 1. Kiểm tra train và nguồn

Chỉ đọc `data/verified/qa/train/LEGAL_RETRIEVAL_TRAIN.jsonl` ở bước chọn cấu hình. Mapping gold được dùng để **chấm điểm**, không đi vào query hay cơ sở tri thức.

In [3]:
qa_train = discover_qa(bundle, split="train")
assert len(qa_train) == 320
assert len({c["case_family_id"] for c in qa_train}) == len(qa_train)
assert all(c["canonical_groups"] for c in qa_train)
train_hash = digest(qa_train)
source_by_id = {r["id"]: r for r in bundle["records"]}
train_rows = []
for case in qa_train:
    primary = [g["any_of"][0] for g in case["canonical_groups"]]
    train_rows.append(
        {
            "case_type": case.get("case_type", ""),
            "section_scope": ",".join(case.get("section_scope", [])),
            "primary_tariff": any(source_by_id[s]["chunk_type"] == "tariff_entry" for s in primary),
            "primary_count": len(primary),
        }
    )
print("Train:", len(qa_train), "unique queries:", len({c["query"].casefold() for c in qa_train}))
display(
    pd.DataFrame(train_rows)
    .groupby(["case_type", "primary_tariff"])
    .size()
    .rename("count")
    .reset_index()
)
print("Train hash:", train_hash)

Train: 320 unique queries: 320


,case_type,primary_tariff,count
0,cross_section,False,1
1,cross_section,True,6
2,gir,False,31
3,legal_rule,False,9
4,note_exclusion,False,2
5,note_rule,False,80
6,tariff_direct,True,90
7,tariff_plus_note,True,8
8,tt85,False,93


Train hash: 72835489c3b4a413b56e8f379323e91371b1c207404284bebd0fa2ab91370358


## 2. Chọn retriever và trọng số trên train

Thử các họ Dense/Sparse/BM25/FTS/Exact và Weighted RRF như notebook 02. Chọn theo tỷ lệ **đủ mọi căn cứ primary trong Top-10**, rồi Recall và MRR. Đây là chọn retrieval policy, không fine-tune BGE-M3.

In [4]:
with capture_output():
    train_cache = branch_cache(
        runtime,
        qa_train,
        intent="knowledge",
        pool=100,
        filters={"chunk_types": QA_CHUNK_TYPES},
        mask_questioned_codes=True,
    )
family_best = []
for family in CONFIGURATIONS:
    family_best.append(sweep_qa_weights(train_cache, family)[0])
base_choice = max(family_best, key=qa_objective)
base_policy = {
    "configuration": base_choice["configuration"],
    "weights": base_choice["weights"],
    "k": base_choice["k"],
    "pool": 100,
    "reranker": "cross_encoder",
    "rerank_pool": 30,
    "rank_weight": 0.75,
    "graph_before_rerank": False,
    "query_split_weight": 0.0,
}
old_policy = {
    "configuration": "C11",
    "weights": {"dense": 0.30, "sparse": 0.05, "bm25": 0.30, "fts": 0.05, "exact": 0.30},
    "k": 5,
    "pool": 100,
    "reranker": "cross_encoder",
    "rerank_pool": 30,
    "rank_weight": 0.75,
    "graph_before_rerank": False,
    "query_split_weight": 0.0,
}
display(
    pd.DataFrame(family_best)[
        [
            "configuration",
            "weights",
            "k",
            "all_groups@5",
            "all_groups@10",
            "group_recall@10",
            "mrr@10",
        ]
    ]
)
print("Train-selected base:", base_policy)
print("Previous snapshot baseline is evaluated as a transfer, not assumed optimal.")

,configuration,weights,k,all_groups@5,all_groups@10,group_recall@10,mrr@10
0,C1,{'dense': 1.0},5,0.731250,0.800000,0.814688,0.616851
1,C2,{'sparse': 1.0},5,0.671875,0.781250,0.792812,0.603230
2,C3,{'bm25': 1.0},5,0.671875,0.778125,0.796979,0.574707
3,C4,{'fts': 1.0},5,0.206250,0.253125,0.270313,0.138206
4,C5,"{'dense': 0.5056255836134853, 'sparse': 0.4943744163865147}",20,0.750000,0.834375,0.844896,0.643437
5,C6,"{'dense': 0.6120720513669394, 'bm25': 0.3879279486330606}",60,0.775000,0.859375,0.873021,0.656286
6,C7,"{'dense': 0.47033777611194416, 'sparse': 0.38669423915801865, 'fts': 0.1429679847300372}",20,0.753125,0.837500,0.851146,0.641551
7,C8,"{'dense': 0.35, 'sparse': 0.25, 'exact': 0.4}",20,0.771875,0.853125,0.860521,0.649844
8,C9,"{'dense': 0.4, 'bm25': 0.2, 'exact': 0.4}",20,0.784375,0.868750,0.878229,0.659363
9,C10,"{'dense': 0.25, 'sparse': 0.2, 'bm25': 0.2, 'fts': 0.05, 'exact': 0.3}",60,0.759375,0.878125,0.888125,0.656147


Train-selected base: {'configuration': 'C10', 'weights': {'dense': 0.25, 'sparse': 0.2, 'bm25': 0.2, 'fts': 0.05, 'exact': 0.3}, 'k': 60, 'pool': 100, 'reranker': 'cross_encoder', 'rerank_pool': 30, 'rank_weight': 0.75, 'graph_before_rerank': False, 'query_split_weight': 0.0}
Previous snapshot baseline is evaluated as a transfer, not assumed optimal.


## 3. Ablation trên 320 train

Giữ cùng trọng số đã chọn và thay vị trí graph, kích thước pool, tỷ lệ giữ ứng viên trực tiếp, mức pha thứ hạng gốc và truy vấn bỏ mã nghi vấn. Cả heading/Notes mới đều phải là chunks có văn bản nguồn và locator. Latency bên dưới đo trên mô hình đã nạp, cache reranker có thể làm lần sau nhanh hơn; không dùng latency này để chọn chất lượng.

In [5]:
variants = {
    "baseline_old_snapshot": old_policy,
    "direct_30": base_policy,
    "direct_50": {**base_policy, "rerank_pool": 50},
    "graph_30_70": {
        **base_policy,
        "graph_before_rerank": True,
        "rerank_pool": 30,
        "direct_fraction": 0.70,
    },
    "graph_50_50": {
        **base_policy,
        "graph_before_rerank": True,
        "rerank_pool": 50,
        "direct_fraction": 0.50,
    },
    "graph_50_70": {
        **base_policy,
        "graph_before_rerank": True,
        "rerank_pool": 50,
        "direct_fraction": 0.70,
    },
    "graph_50_85": {
        **base_policy,
        "graph_before_rerank": True,
        "rerank_pool": 50,
        "direct_fraction": 0.85,
    },
    "graph_100_70": {
        **base_policy,
        "graph_before_rerank": True,
        "rerank_pool": 100,
        "direct_fraction": 0.70,
    },
    "graph_50_pure": {
        **base_policy,
        "graph_before_rerank": True,
        "rerank_pool": 50,
        "direct_fraction": 0.70,
        "rank_weight": 1.0,
    },
    "graph_50_neutral_25": {
        **base_policy,
        "graph_before_rerank": True,
        "rerank_pool": 50,
        "direct_fraction": 0.70,
        "query_split_weight": 0.25,
    },
    "graph_50_neutral_50": {
        **base_policy,
        "graph_before_rerank": True,
        "rerank_pool": 50,
        "direct_fraction": 0.70,
        "query_split_weight": 0.50,
    },
}

with capture_output():
    runtime.qa_search(qa_train[0]["query"], policy_override=base_policy, include_package=False)
train_experiments = []
for name, policy in variants.items():
    started = time.perf_counter()
    for case in qa_train:
        with capture_output():
            result = runtime.qa_search(case["query"], policy_override=policy, include_package=False)
        metric = qa_direct_coverage(case, result["matches"])
        required = {g["any_of"][0] for g in case["canonical_groups"]}
        pool_ids = set(result["pre_rerank_ids"])
        train_experiments.append(
            {
                "variant": name,
                "case": case["id"],
                "case_type": case.get("case_type", ""),
                "section_scope": ",".join(case.get("section_scope", [])),
                "pool_complete": required <= pool_ids,
                "pool_recall": len(required & pool_ids) / len(required),
                "all_groups@5": metric["all_groups@5"],
                "all_groups@10": metric["all_groups@10"],
                "group_recall@10": metric["group_recall@10"],
                "mrr@10": metric["mrr@10"],
                "elapsed_ms": result["elapsed_ms"],
                "query_split_used": result["query_split_used"],
            }
        )
    print(name, "seconds:", round(time.perf_counter() - started, 1), flush=True)
train_frame = pd.DataFrame(train_experiments)
train_summary = (
    train_frame.groupby("variant")
    .agg(
        count=("case", "count"),
        pool_complete=("pool_complete", "mean"),
        pool_recall=("pool_recall", "mean"),
        **{
            "all_groups@5": ("all_groups@5", "mean"),
            "all_groups@10": ("all_groups@10", "mean"),
            "group_recall@10": ("group_recall@10", "mean"),
            "mrr@10": ("mrr@10", "mean"),
        },
        median_ms=("elapsed_ms", "median"),
        split_queries=("query_split_used", "sum"),
    )
    .reset_index()
)
display(train_summary.sort_values(["all_groups@10", "group_recall@10", "mrr@10"], ascending=False))
display(
    train_frame.groupby(["variant", "case_type"])["all_groups@10"]
    .agg(["mean", "count"])
    .reset_index()
)


baseline_old_snapshot seconds: 50.2


direct_30 seconds: 51.4


direct_50 seconds: 49.5


graph_30_70 seconds: 48.5


graph_50_50 seconds: 47.9


...

graph_50_pure seconds: 49.7


graph_50_neutral_25 seconds: 52.0


graph_50_neutral_50 seconds: 50.9


,variant,count,pool_complete,pool_recall,all_groups@5,all_groups@10,group_recall@10,mrr@10,median_ms,split_queries
0,baseline_old_snapshot,320,0.918750,0.929792,0.806250,0.871875,0.881875,0.723390,112.563041,0
10,graph_50_pure,320,0.943750,0.946979,0.796875,0.868750,0.873021,0.721317,110.848146,0
4,graph_30_70,320,0.934375,0.941250,0.812500,0.865625,0.875625,0.718690,108.957687,0
6,graph_50_70,320,0.943750,0.946979,0.812500,0.865625,0.874062,0.718842,101.117313,0
7,graph_50_85,320,0.943750,0.946979,0.812500,0.865625,0.874062,0.718786,102.285708,0
8,graph_50_neutral_25,320,0.943750,0.946979,0.812500,0.865625,0.874062,0.715196,109.487375,5
5,graph_50_50,320,0.934375,0.941250,0.812500,0.862500,0.872500,0.718864,101.269750,0
2,direct_50,320,0.937500,0.940729,0.812500,0.862500,0.870937,0.718265,111.278542,0
3,graph_100_70,320,0.953125,0.956563,0.812500,0.862500,0.870937,0.717484,104.404437,0
1,direct_30,320,0.928125,0.932917,0.809375,0.859375,0.869375,0.717718,112.560584,0


,variant,case_type,mean,count
0,baseline_old_snapshot,cross_section,0.428571,7
1,baseline_old_snapshot,gir,0.967742,31
2,baseline_old_snapshot,legal_rule,0.333333,9
3,baseline_old_snapshot,note_exclusion,1.000000,2
4,baseline_old_snapshot,note_rule,0.975000,80
...,...,...,...,...
83,graph_50_pure,note_exclusion,1.000000,2
84,graph_50_pure,note_rule,0.950000,80
85,graph_50_pure,tariff_direct,0.722222,90
86,graph_50_pure,tariff_plus_note,1.000000,8


## 4. Đóng băng QA policy trước khi mở test

Lấy phương án tốt nhất trên train trong phạm vi đã thử. Nếu graph không thắng, nó vẫn được báo trong bảng test để chẩn đoán, nhưng không được âm thầm đưa vào runtime.

In [6]:
candidate_names = list(variants)
new_candidate_names = [name for name in variants if name != "baseline_old_snapshot"]
quality = {name: train_summary.set_index("variant").loc[name] for name in candidate_names}
chosen_name = max(
    candidate_names,
    key=lambda name: (
        quality[name]["all_groups@10"],
        quality[name]["group_recall@10"],
        quality[name]["all_groups@5"],
        quality[name]["mrr@10"],
    ),
)
selected_policy = {
    **variants[chosen_name],
    "selection_status": "QA_320_TRAIN_ONLY",
    "train_sha256": train_hash,
    "train_count": len(qa_train),
    "tested_variants": len(variants),
    "selected_variant": chosen_name,
    "objective": ["all_groups@10", "group_recall@10", "all_groups@5", "mrr@10"],
    "candidate_types": QA_CHUNK_TYPES,
    "limitations": "Best among tested policies on supplied train; no model fine-tuning.",
}
hs_selection = read(ROOT / "config/selected_retrieval.json")
assert hs_selection["snapshot"] == runtime.snapshot and hs_selection["profile"] == runtime.profile
selected_config = {**hs_selection, "qa": selected_policy}
assert {k: v for k, v in selected_config.items() if k != "qa"} == {
    k: v for k, v in hs_selection.items() if k != "qa"
}
write_json(ROOT / "config/selected_retrieval.json", selected_config)
frozen_hash = digest(selected_config)
print("Frozen before test:", chosen_name, selected_policy)
print("Configuration digest:", frozen_hash)

Frozen before test: baseline_old_snapshot {'configuration': 'C11', 'weights': {'dense': 0.3, 'sparse': 0.05, 'bm25': 0.3, 'fts': 0.05, 'exact': 0.3}, 'k': 5, 'pool': 100, 'reranker': 'cross_encoder', 'rerank_pool': 30, 'rank_weight': 0.75, 'graph_before_rerank': False, 'query_split_weight': 0.0, 'selection_status': 'QA_320_TRAIN_ONLY', 'train_sha256': '72835489c3b4a413b56e8f379323e91371b1c207404284bebd0fa2ab91370358', 'train_count': 320, 'tested_variants': 11, 'selected_variant': 'baseline_old_snapshot', 'objective': ['all_groups@10', 'group_recall@10', 'all_groups@5', 'mrr@10'], 'candidate_types': ['gir_rule', 'gir_explanation', 'section_note', 'chapter_note', 'subheading_note', 'tariff_entry', 'legal_clause'], 'limitations': 'Best among tested policies on supplied train; no model fine-tuning.'}
Configuration digest: 1af57c027a7fc22ae6f6afaa21baaae62bf78ecc1d42016231e1b7d9113c7e23


## 5. Test 150 câu sau khi đóng băng

So sánh cách cũ, chính sách tìm trực tiếp, graph tốt nhất trên train, và policy đã chọn. Báo riêng 50 câu cũ đã từng được dùng để chẩn đoán; không dùng test để đổi trọng số hay chọn lại policy.

In [7]:
assert digest(read(ROOT / "config/selected_retrieval.json")) == frozen_hash
qa_test = discover_qa(bundle, split="test")
assert len(qa_test) == 150
assert not ({c["case_family_id"] for c in qa_train} & {c["case_family_id"] for c in qa_test})
train_best_direct = max(
    (n for n in new_candidate_names if not variants[n]["graph_before_rerank"]),
    key=lambda n: (
        quality[n]["all_groups@10"],
        quality[n]["group_recall@10"],
        quality[n]["mrr@10"],
    ),
)
train_best_graph = max(
    (n for n in new_candidate_names if variants[n]["graph_before_rerank"]),
    key=lambda n: (
        quality[n]["all_groups@10"],
        quality[n]["group_recall@10"],
        quality[n]["mrr@10"],
    ),
)
test_variants = dict.fromkeys(
    ["baseline_old_snapshot", train_best_direct, train_best_graph, chosen_name]
)
test_rows = []
for name in test_variants:
    policy = variants[name]
    started = time.perf_counter()
    for case in qa_test:
        with capture_output():
            result = runtime.qa_search(case["query"], policy_override=policy, include_package=True)
        metric = qa_direct_coverage(case, result["matches"])
        required = {g["any_of"][0] for g in case["canonical_groups"]}
        pool_ids = set(result["pre_rerank_ids"])
        package_ids = {r["id"] for r in result["package"]}
        test_rows.append(
            {
                "variant": name,
                "case": case["id"],
                "query": case["query"],
                "source_group": "old_50_diagnostic"
                if case.get("legacy_original_index") is not None
                else "new_100",
                "case_type": case.get("case_type", ""),
                "section_scope": ",".join(case.get("section_scope", [])),
                "pool_complete": required <= pool_ids,
                "pool_recall": len(required & pool_ids) / len(required),
                "all_groups@5": metric["all_groups@5"],
                "all_groups@10": metric["all_groups@10"],
                "group_recall@10": metric["group_recall@10"],
                "mrr@10": metric["mrr@10"],
                "package_complete": required <= package_ids,
                "package_recall": len(required & package_ids) / len(required),
                "missing_primary": sorted(required - {r["id"] for r in result["matches"]}),
                "returned_top10": [r["id"] for r in result["matches"]],
                "elapsed_ms": result["elapsed_ms"],
                "query_split_used": result["query_split_used"],
            }
        )
    print("TEST", name, "seconds:", round(time.perf_counter() - started, 1), flush=True)
test_frame = pd.DataFrame(test_rows)
columns = [
    "pool_complete",
    "pool_recall",
    "all_groups@5",
    "all_groups@10",
    "group_recall@10",
    "mrr@10",
    "package_complete",
    "package_recall",
]
test_summary = (
    test_frame.groupby("variant")[columns]
    .mean()
    .join(test_frame.groupby("variant").elapsed_ms.agg(["median", lambda s: s.quantile(0.95)]))
)
test_summary.columns = columns + ["latency_p50_ms", "latency_p95_ms"]
display(test_summary)
display(test_frame.groupby(["source_group", "variant"])[columns].mean())
display(
    test_frame.groupby(["case_type", "variant"])[
        ["all_groups@10", "group_recall@10", "package_complete"]
    ].mean()
)
print(
    "Frozen policy digest unchanged:",
    digest(read(ROOT / "config/selected_retrieval.json")) == frozen_hash,
)

TEST baseline_old_snapshot seconds: 116.1


TEST direct_50 seconds: 155.6


TEST graph_50_pure seconds: 89.0


,pool_complete,pool_recall,all_groups@5,all_groups@10,group_recall@10,mrr@10,package_complete,package_recall,latency_p50_ms,latency_p95_ms
variant,,,,,,,,,,
baseline_old_snapshot,0.880000,0.886667,0.720000,0.806667,0.821111,0.656693,0.913333,0.920000,928.159563,1668.264798
direct_50,0.906667,0.913333,0.713333,0.806667,0.824444,0.648447,0.913333,0.916667,984.766458,1510.189435
graph_50_pure,0.933333,0.936667,0.726667,0.800000,0.817778,0.657246,0.913333,0.920000,658.648646,1072.162796


pool_complete  pool_recall  \
source_group      variant                                             
new_100           baseline_old_snapshot           0.89         0.89   
                  direct_50                       0.92         0.92   
                  graph_50_pure                   0.93         0.93   
old_50_diagnostic baseline_old_snapshot           0.86         0.88   
                  direct_50                       0.88         0.90   
                  graph_50_pure                   0.94         0.95   

                                         all_groups@5  all_groups@10  \
source_group      variant                                              
new_100           baseline_old_snapshot          0.77           0.86   
                  direct_50                      0.78           0.85   
                  graph_50_pure                  0.79           0.85   
old_50_diagnostic baseline_old_snapshot          0.62           0.70   
                  direct_50                      0.58           0.72   
                  graph_50_pure                  0.60           0.70   

                                         group_recall@10    mrr@10  \
source_group      variant                                            
new_100           baseline_old_snapshot         0.865000  0.729694   
                  direct_50                     0.860000  0.732552   
                  graph_50_pure                 0.860000  0.728833   
old_50_diagnostic baseline_old_snapshot         0.733333  0.510690   
                  direct_50                     0.753333  0.480238   
                  graph_50_pure                 0.733333  0.514071   

                                         package_complete  package_recall  
source_group      variant                                                  
new_100           baseline_old_snapshot              0.91           0.915  
                  direct_50                          0.91           0.915  
                  graph_50_pure                      0.91           0.915  
old_50_diagnostic baseline_old_snapshot              0.92           0.930  
                  direct_50                          0.92           0.920  
                  graph_50_pure                      0.92           0.930

all_groups@10  group_recall@10  \
case_type        variant                                                 
cross_section    baseline_old_snapshot       0.333333         0.500000   
                 direct_50                   0.333333         0.500000   
                 graph_50_pure               0.333333         0.500000   
gir              baseline_old_snapshot       0.937500         0.979167   
                 direct_50                   0.937500         0.979167   
                 graph_50_pure               0.937500         0.979167   
legal_rule       baseline_old_snapshot       1.000000         1.000000   
                 direct_50                   1.000000         1.000000   
                 graph_50_pure               1.000000         1.000000   
note_exclusion   baseline_old_snapshot       1.000000         1.000000   
                 direct_50                   1.000000         1.000000   
                 graph_50_pure               1.000000         1.000000   
note_rule        baseline_old_snapshot       1.000000         1.000000   
                 direct_50                   1.000000         1.000000   
                 graph_50_pure               1.000000         1.000000   
tariff_direct    baseline_old_snapshot       0.702703         0.702703   
                 direct_50                   0.716216         0.716216   
                 graph_50_pure               0.702703         0.702703   
tariff_plus_note baseline_old_snapshot       0.846154         0.884615   
                 direct_50                   0.807692         0.865385   
                 graph_50_pure               0.807692         0.865385   
tt85             baseline_old_snapshot       1.000000         1.000000   
                 direct_50                   1.000000         1.000000   
                 graph_50_pure               1.000000         1.000000   

                                        package_complete  
case_type        variant                                  
cross_section    baseline_old_snapshot          0.666667  
                 direct_50                      0.666667  
                 graph_50_pure                  0.666667  
gir              baseline_old_snapshot          1.000000  
                 direct_50                      1.000000  
                 graph_50_pure                  1.000000  
legal_rule       baseline_old_snapshot          1.000000  
                 direct_50                      1.000000  
                 graph_50_pure                  1.000000  
note_exclusion   baseline_old_snapshot          1.000000  
                 direct_50                      1.000000  
                 graph_50_pure                  1.000000  
note_rule        baseline_old_snapshot          1.000000  
                 direct_50                      1.000000  
                 graph_50_pure                  1.000000  
tariff_direct    baseline_old_snapshot          0.864865  
                 direct_50                      0.864865  
                 graph_50_pure                  0.864865  
tariff_plus_note baseline_old_snapshot          0.923077  
                 direct_50                      0.923077  
                 graph_50_pure                  0.923077  
tt85             baseline_old_snapshot          1.000000  
                 direct_50                      1.000000  
                 graph_50_pure                  1.000000

Frozen policy digest unchanged: True


In [8]:
base_test = test_frame[test_frame.variant == train_best_direct].set_index("case")
selected_test = test_frame[test_frame.variant == chosen_name].set_index("case")
comparison = selected_test[
    ["query", "source_group", "case_type", "section_scope", "missing_primary", "returned_top10"]
].copy()
comparison["direct_all@10"] = base_test["all_groups@10"]
comparison["selected_all@10"] = selected_test["all_groups@10"]
comparison["change"] = comparison["selected_all@10"].astype(int) - comparison[
    "direct_all@10"
].astype(int)
print(
    "Gains:",
    (comparison.change > 0).sum(),
    "losses:",
    (comparison.change < 0).sum(),
    "unchanged:",
    (comparison.change == 0).sum(),
)
display(comparison.loc[comparison.change != 0].head(30))
display(
    test_summary[["pool_complete", "all_groups@10", "package_complete"]]
    .style.format("{:.1%}")
    .bar(axis=0, color="#87a9d8", vmin=0, vmax=1)
    .set_caption("QA 150 test: đủ căn cứ trước rerank / Top-10 / gói căn cứ")
)


Gains: 1 losses: 1 unchanged: 148


,query,source_group,case_type,section_scope,missing_primary,returned_top10,direct_all@10,selected_all@10,change
case,,,,,,,,,
LQ026,Dây dẫn điện cách điện đã lắp đầu nối điện thuộc phân nhóm nào?,old_50_diagnostic,tariff_direct,XVI,[tt31:SXVI-T-854442],"[tt31:SXVI-T-85444219, tt31:SXVI-T-85444296, tt31:SXVI-T-85444229, tt31:SXVI-T-85444295, tt31:SXVI-T-85444299, tt31:SXVI-T-85444233, tt3...",True,False,-1
LQ115,"Thiết bị điều khiển từ xa hồng ngoại không dây có thuộc 85.37 không, hay phải xem xét 85.43? Truy xuất chú giải và Heading 85.43.",new_100,tariff_plus_note,XVI,[],"[tt31:SXVI-N-CH85_NOTE_10, tt31:SXVI-T-85437021, tt31:SXVI-T-85176243, tt31:SXVI-T-85381021, tt31:SXVI-N-SXVI_NOTE_2B, tt31:SXVI-T-85176...",False,True,1


,pool_complete,all_groups@10,package_complete
variant,,,
baseline_old_snapshot,88.0%,80.7%,91.3%
direct_50,90.7%,80.7%,91.3%
graph_50_pure,93.3%,80.0%,91.3%


## 6. Runtime và kiểm tra

Đối chiếu runtime dùng cấu hình đã đóng băng với đường thử nghiệm. Kiểm tra chính sách HS không bị thay đổi khi thêm khóa QA; kết quả HS và QA thuộc cùng snapshot.

In [9]:
assert digest(read(ROOT / "config/selected_retrieval.json")) == frozen_hash
with connect() as fresh_conn:
    fresh = Retriever(fresh_conn, embedder)
    assert fresh.qa_policy["selected_variant"] == chosen_name
    for case in qa_train[:5] + qa_test[:5]:
        with capture_output():
            expected = runtime.qa_search(
                case["query"], policy_override=selected_policy, include_package=False
            )
            actual = fresh.qa_search(case["query"], include_package=False)
        assert [r["id"] for r in expected["matches"]] == [r["id"] for r in actual["matches"]]
print("QA runtime parity: 10/10 exact Top-10.")
assert {k: v for k, v in read(ROOT / "config/selected_retrieval.json").items() if k != "qa"} == {
    k: v for k, v in hs_selection.items() if k != "qa"
}
print("HS configuration unchanged by QA selection.")

QA runtime parity: 10/10 exact Top-10.
HS configuration unchanged by QA selection.


In [ ]:
import subprocess

validation_env = {**os.environ, "RAG_RUN_DB_TESTS": "1", "TOKENIZERS_PARALLELISM": "false"}
for command in [
    [str(ROOT / ".venv/bin/ruff"), "check", "rag", "tests", "notebooks"],
    [str(ROOT / ".venv/bin/mypy"), "rag"],
    [str(ROOT / ".venv/bin/bandit"), "-r", "rag", "-ll", "-q"],
    [str(ROOT / ".venv/bin/python"), "-m", "pytest", "tests", "-q"],
    ["git", "diff", "--check"],
]:
    checked = subprocess.run(
        command, cwd=ROOT, env=validation_env, capture_output=True, text=True, check=False
    )
    print(
        " ".join(command),
        "exit",
        checked.returncode,
        checked.stdout[-700:] or checked.stderr[-700:] or "PASS",
    )
    assert checked.returncode == 0
assert digest(read(ROOT / "config/selected_retrieval.json")) == frozen_hash
conn.close()
print("Completed: snapshot", bundle["snapshot"], "QA policy", chosen_name)

/Users/capkimkhanh/Documents/DUT4_1/CSHTTT_Project/frappe-bench/apps/aiAgent/.venv/bin/ruff check rag tests notebooks exit 0 All checks passed!



/Users/capkimkhanh/Documents/DUT4_1/CSHTTT_Project/frappe-bench/apps/aiAgent/.venv/bin/mypy rag exit 0 Success: no issues found in 6 source files



/Users/capkimkhanh/Documents/DUT4_1/CSHTTT_Project/frappe-bench/apps/aiAgent/.venv/bin/bandit -r rag -ll -q exit 0 PASS


/Users/capkimkhanh/Documents/DUT4_1/CSHTTT_Project/frappe-bench/apps/aiAgent/.venv/bin/python -m pytest tests -q exit 0 .....................................................                    [100%]
53 passed in 23.64s

git diff --check exit 0 PASS
Completed: snapshot 1a67df40d28f8b07a668746c QA policy baseline_old_snapshot
